# Análisis complementario: aislamiento del aporte del Análisis de Datos Funcionales

Este cuaderno separa el efecto de la representación funcional del efecto asociado al rango
espectral, la resolución en frecuencia y la reducción de dimensión. Para ello compara, con el
mismo clasificador y las mismas particiones, representaciones espectrales resumidas,
discretizadas y funcionales (análisis de ablación).

Se utiliza **regresión logística** bajo el mismo esquema de validación por sujeto empleado en el
modelado principal: `StratifiedGroupKFold`, 5 pliegues y semillas 42, 123 y 2024. Todas las
transformaciones dependientes de los datos se ajustan **solo con el conjunto de entrenamiento de
cada pliegue**.

Se comparan las siguientes representaciones:

1. **Escalar theta + alfa + PAF:** 20 potencias relativas logarítmicas (theta y alfa en 10 canales) + una PAF posterior por sujeto.
2. **PSD discreta 4–13 Hz:** vector de `log10(PSD relativa)` sin base ni suavizado, con 19 frecuencias × 10 canales = 190 variables.
3. **PSD discreta 4–13 Hz + PCA:** control de reducción dimensional multivariada, ajustada dentro del entrenamiento.
4. **A1 B-spline:** representación funcional original de 11 coeficientes × 10 canales.
5. **A1 Cholesky:** misma función B-spline expresada en coordenadas coherentes con la geometría funcional L2.
6. **A1 Cholesky + FPCA:** reducción funcional ajustada por canal dentro de cada pliegue.

El objetivo no es seleccionar una nueva configuración, sino **cuantificar de manera descriptiva
qué parte del desempeño puede asociarse a la representación funcional frente a controles
espectrales emparejados**.

Entradas: salidas de los cuadernos 01 (`epochs_manifest.csv`, `paf_por_sujeto.csv`) y 02A
(`enfoque_A_sujeto_dataset.npz`).

In [ ]:
# Rutas portátiles: Google Drive solo se monta si USE_GOOGLE_DRIVE=1 (Colab).
import os
from pathlib import Path

if os.environ.get('USE_GOOGLE_DRIVE') == '1':
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
else:
    print('Ejecución local: se usan PARKINSON_EEG_ROOT y PARKINSON_EEG_DATA.')

In [ ]:
# ============================================================================
# 1. ENTORNO, CONFIGURACIÓN Y REPRODUCIBILIDAD
# ============================================================================
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import logging
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from scipy.integrate import simpson
from scipy.signal import welch
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)
LOGGER = logging.getLogger('aporte_ADF')

CONFIG: dict[str, Any] = {
    'random_state': 42,
    'seeds': [42, 123, 2024],
    'n_folds': 5,
    'min_epochs': 6,
    'psd_fmin_hz': 1.0,
    'psd_fmax_hz': 40.0,
    'normalization_fmax_hz': 30.0,
    'functional_fmin_hz': 4.0,
    'functional_fmax_hz': 13.0,
    'welch_window_seconds': 2.0,
    'welch_overlap_fraction': 0.5,
    'eps': 1e-20,
    'pca_variance_threshold': 0.95,
    'logreg_C': 1.0,
    'logreg_max_iter': 2000,
    'expected_channels': ['C3', 'Cz', 'C4', 'P7', 'P8', 'P3', 'Pz', 'P4', 'O1', 'O2'],
}

np.random.seed(CONFIG['random_state'])

import os  # rutas portátiles (ver docs/reproducibility.md)
_REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FINAL_ROOT = Path(os.environ.get('PARKINSON_EEG_ROOT', _REPO_ROOT / 'artifacts')).expanduser().resolve()
PREPROC_DIR = FINAL_ROOT / '01_preprocesamiento'
A_DIR = FINAL_ROOT / '02A_enfoque_A'
OUT_DIR = FINAL_ROOT / '04_aporte_ADF'
FIG_DIR = OUT_DIR / 'figuras'

MANIFEST_PATH = PREPROC_DIR / 'epochs_manifest.csv'
PAF_PATH = PREPROC_DIR / 'descriptiva_preproc' / 'paf_por_sujeto.csv'
A1_PATH = A_DIR / 'enfoque_A_sujeto_dataset.npz'

# Validar primero las entradas; solo después crear la carpeta de salida.
REQUIRED_INPUTS = [MANIFEST_PATH, PAF_PATH, A1_PATH]
missing_inputs = [p for p in REQUIRED_INPUTS if not p.exists()]
if missing_inputs:
    print('\nArchivos requeridos no encontrados:')
    for p in missing_inputs:
        print(' -', p)
    print('\nDirectorio raíz esperado:', FINAL_ROOT)
    print('¿Existe la raíz?:', FINAL_ROOT.exists())
    print('¿Existe 01_preprocesamiento?:', PREPROC_DIR.exists())
    print('¿Existe 02A_enfoque_A?:', A_DIR.exists())
    raise FileNotFoundError(
        'No se encontraron todas las salidas de los cuadernos 01 y 02A. '
        'Verifique que esos cuadernos se hayan ejecutado y guardado sus salidas en FINAL_ROOT.'
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = OUT_DIR / 'config_aporte_ADF.json'
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')

print('Salida:', OUT_DIR)
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)


## 2. Carga y alineación de la cohorte

La unidad de evaluación es el **sujeto**. Se utilizan únicamente los 48 sujetos que superaron el
control de calidad del cuaderno 01: 24 HC y 24 PD_OFF.

La representación A1 se carga desde el dataset generado por el cuaderno 02A. El orden de los
sujetos se verifica contra el `epochs_manifest.csv` antes de ejecutar cualquier modelo.

In [ ]:
# ============================================================================
# 2. CARGA Y VALIDACIÓN DE COHORTE
# ============================================================================
for path in [MANIFEST_PATH, PAF_PATH, A1_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'No existe el archivo requerido: {path}')

manifest = pd.read_csv(MANIFEST_PATH)
manifest['error'] = manifest.get('error', '').fillna('').astype(str)
manifest = manifest[
    (manifest['error'] == '')
    & (manifest['n_epocas_conservadas'] >= CONFIG['min_epochs'])
    & (manifest['condition'].isin(['CTL', 'PD_OFF']))
].copy()
manifest['label'] = manifest['condition'].map({'CTL': 0, 'PD_OFF': 1})
manifest['grupo'] = manifest['condition'].map({'CTL': 'HC', 'PD_OFF': 'PD_OFF'})
manifest = manifest.reset_index(drop=True)

assert len(manifest) == 48
assert manifest['subject_id'].nunique() == 48
assert manifest['label'].value_counts().sort_index().to_dict() == {0: 24, 1: 24}

a1 = np.load(A1_PATH, allow_pickle=False)
X_A1_RAW = np.asarray(a1['X_raw'], dtype=float)
X_A1_CHOL = np.asarray(a1['X_chol'], dtype=float)
y_a1 = np.asarray(a1['y'], dtype=int)
A_CHANNELS = [str(x) for x in a1['channels'].tolist()]
A_FREQS = np.asarray(a1['freqs'], dtype=float)

subject_ids = manifest['subject_id'].astype(str).to_numpy()
y = manifest['label'].to_numpy(dtype=int)
# Para reproducir exactamente las particiones de A1 del cuaderno de modelado se conservan
# los identificadores de grupo almacenados por el cuaderno 02A.
groups_cv = np.asarray(a1['groups'])
assert len(np.unique(groups_cv)) == 48

assert X_A1_RAW.shape == (48, 10, 11), X_A1_RAW.shape
assert X_A1_CHOL.shape == X_A1_RAW.shape
assert np.array_equal(y_a1, y), (
    'El orden/etiquetas del dataset A1 no coincide con el manifest actual. '
    'No continúe hasta resolver la alineación.'
)
assert A_CHANNELS == CONFIG['expected_channels'], (A_CHANNELS, CONFIG['expected_channels'])

print('Cohorte validada:')
print(manifest.groupby('grupo')['subject_id'].nunique())
print('\nA1 X_raw:', X_A1_RAW.shape)
print('A1 X_chol:', X_A1_CHOL.shape)
print('Canales:', A_CHANNELS)
print('Frecuencias A:', A_FREQS)


## 3. Construcción de los controles espectrales

### 3.1 PSD discreta 4–13 Hz

Este control utiliza **exactamente el objeto previo a la proyección B-spline de A1**: para cada época y canal se estima Welch, se normaliza por la potencia total 1–30 Hz, se aplica `log10` y luego se promedian las épocas del sujeto. Finalmente se conservan los 19 puntos entre 4 y 13 Hz. No se utiliza base funcional ni suavizado.

### 3.2 Escalares theta + alfa + PAF

Para que este control sea una versión restringida de la línea base escalar del estudio, theta y alfa se calculan siguiendo el mismo orden de operaciones del baseline N0: **promedio lineal de la PSD entre épocas → integración por banda → potencia relativa respecto a 1–30 Hz → log10**. Se obtienen dos valores por canal (20 variables) y se añade una única PAF posterior por sujeto calculada mediante centro de gravedad en el cuaderno 01 (21 variables en total).

In [ ]:
# ============================================================================
# 3. CONSTRUCCIÓN DE PSD DISCRETA Y ESCALARES THETA+ALFA+PAF
# ============================================================================
PSD_FMIN = CONFIG['psd_fmin_hz']
PSD_FMAX = CONFIG['psd_fmax_hz']
NORM_FMAX = CONFIG['normalization_fmax_hz']
BAND_FMIN = CONFIG['functional_fmin_hz']
BAND_FMAX = CONFIG['functional_fmax_hz']
WELCH_SEC = CONFIG['welch_window_seconds']
WELCH_OVL = CONFIG['welch_overlap_fraction']
EPS = CONFIG['eps']


def psd_rel_log_epochs(data_uv: np.ndarray, sfreq: float) -> tuple[np.ndarray, np.ndarray]:
    """Replica la transformación espectral previa a B-spline del Enfoque A."""
    nperseg = min(int(sfreq * WELCH_SEC), data_uv.shape[-1])
    noverlap = int(nperseg * WELCH_OVL)
    freqs, psd = welch(
        data_uv,
        fs=sfreq,
        nperseg=nperseg,
        noverlap=noverlap,
        axis=-1,
    )
    norm_mask = (freqs >= PSD_FMIN) & (freqs <= NORM_FMAX)
    df = float(freqs[1] - freqs[0])
    total = simpson(psd[..., norm_mask], dx=df, axis=-1)
    total = np.maximum(total, EPS)
    rel_log = np.log10(psd / total[..., None] + EPS)
    band_mask = (freqs >= BAND_FMIN) & (freqs <= BAND_FMAX)
    return rel_log[..., band_mask], freqs[band_mask]


def average_linear_psd(data_uv: np.ndarray, sfreq: float) -> tuple[np.ndarray, np.ndarray]:
    """PSD lineal promedio por sujeto, equivalente al baseline escalar N0."""
    nperseg = min(int(sfreq * WELCH_SEC), data_uv.shape[-1])
    noverlap = int(nperseg * WELCH_OVL)
    freqs, psd = welch(
        data_uv,
        fs=sfreq,
        nperseg=nperseg,
        noverlap=noverlap,
        axis=-1,
    )
    mask = (freqs >= PSD_FMIN) & (freqs <= PSD_FMAX)
    return psd[..., mask].mean(axis=0), freqs[mask]


def theta_alpha_scalar_features(psd_avg: np.ndarray, freqs: np.ndarray) -> np.ndarray:
    """Log10 de potencia relativa theta y alfa por canal: shape (canales, 2)."""
    df = float(freqs[1] - freqs[0])
    total_mask = (freqs >= PSD_FMIN) & (freqs <= NORM_FMAX)
    total = simpson(psd_avg[..., total_mask], dx=df, axis=-1)
    total = np.maximum(total, EPS)

    values = []
    for lo, hi in [(4.0, 8.0), (8.0, 13.0)]:
        mask = (freqs >= lo) & (freqs <= hi)
        power = simpson(psd_avg[..., mask], dx=df, axis=-1) / total
        values.append(np.log10(power + EPS))
    return np.stack(values, axis=-1)


# PAF principal por sujeto (centro de gravedad).
paf = pd.read_csv(PAF_PATH)
rename_paf = {
    'clase': 'grupo',
    'PAF_cog': 'paf_centro_gravedad_hz',
}
paf = paf.rename(columns={k: v for k, v in rename_paf.items() if k in paf.columns})
required_paf = {'subject_id', 'paf_centro_gravedad_hz'}
if not required_paf.issubset(paf.columns):
    raise ValueError(f'Columnas PAF insuficientes. Disponibles: {list(paf.columns)}')
paf_lookup = paf.set_index('subject_id')['paf_centro_gravedad_hz'].to_dict()

X_grid_list: list[np.ndarray] = []
X_theta_alpha_list: list[np.ndarray] = []
Paf_list: list[float] = []
freqs_grid_ref: np.ndarray | None = None

for _, row in manifest.iterrows():
    file_path = Path(row['file'])
    if not file_path.exists():
        raise FileNotFoundError(file_path)

    data = np.load(file_path).astype(np.float64)
    ch_names = str(row['ch_names']).split(',')
    index = {ch: i for i, ch in enumerate(ch_names)}
    missing = [ch for ch in A_CHANNELS if ch not in index]
    if missing:
        raise ValueError(f"{row['subject_id']}: faltan canales {missing}")
    order = [index[ch] for ch in A_CHANNELS]
    data = data[:, order, :]

    # (i) PSD discreta: misma transformación de A1 antes de B-spline.
    rel_log_epochs, freqs_grid = psd_rel_log_epochs(data, float(row['sfreq']))
    X_grid_list.append(rel_log_epochs.mean(axis=0))  # (10, 19)

    if freqs_grid_ref is None:
        freqs_grid_ref = freqs_grid
    else:
        np.testing.assert_allclose(freqs_grid_ref, freqs_grid)

    # (ii) Línea base escalar restringida theta + alfa.
    psd_avg, freqs_full = average_linear_psd(data, float(row['sfreq']))
    X_theta_alpha_list.append(theta_alpha_scalar_features(psd_avg, freqs_full))

    sid = str(row['subject_id'])
    if sid not in paf_lookup:
        raise KeyError(f'No existe PAF para {sid}')
    Paf_list.append(float(paf_lookup[sid]))

X_GRID = np.stack(X_grid_list, axis=0)              # (48, 10, 19)
X_THETA_ALPHA = np.stack(X_theta_alpha_list, axis=0) # (48, 10, 2)
X_PAF = np.asarray(Paf_list, dtype=float)[:, None]   # (48, 1)
X_SCALAR = np.concatenate(
    [X_THETA_ALPHA.reshape(len(X_THETA_ALPHA), -1), X_PAF],
    axis=1,
)                                                     # (48, 21)

assert X_GRID.shape == (48, 10, 19), X_GRID.shape
assert X_THETA_ALPHA.shape == (48, 10, 2), X_THETA_ALPHA.shape
assert X_SCALAR.shape == (48, 21), X_SCALAR.shape
assert np.isfinite(X_GRID).all()
assert np.isfinite(X_SCALAR).all()
assert len(freqs_grid_ref) == 19
np.testing.assert_allclose(freqs_grid_ref, A_FREQS)

FEATURE_PATH = OUT_DIR / 'conjuntos_caracteristicas.npz'
np.savez_compressed(
    FEATURE_PATH,
    X_psd_grid_4_13=X_GRID,
    X_theta_alpha=X_THETA_ALPHA,
    X_paf=X_PAF,
    X_scalar_theta_alpha_paf=X_SCALAR,
    y=y,
    subject_ids=subject_ids,
    channels=np.asarray(A_CHANNELS, dtype=str),
    freqs_4_13=np.asarray(freqs_grid_ref, dtype=float),
)

print('PSD discreta 4–13 Hz:', X_GRID.shape, '->', X_GRID.shape[1] * X_GRID.shape[2], 'variables')
print('Theta + alfa:', X_THETA_ALPHA.shape, '->', X_THETA_ALPHA.shape[1] * X_THETA_ALPHA.shape[2], 'variables')
print('Theta + alfa + PAF:', X_SCALAR.shape)
print('Frecuencias:', freqs_grid_ref)
print('Archivo de características:', FEATURE_PATH)


## 4. Particiones externas comunes

Las seis representaciones se evalúan sobre **exactamente las mismas 15 particiones externas**. Las identidades de entrenamiento y prueba se exportan para trazabilidad.


In [ ]:
# ============================================================================
# 4. GENERACIÓN Y EXPORTACIÓN DE LAS 15 PARTICIONES EXTERNAS
# ============================================================================
splits: list[dict[str, Any]] = []
split_indices: list[tuple[int, int, np.ndarray, np.ndarray]] = []

dummy_X = np.zeros((len(y), 1), dtype=float)
for seed in CONFIG['seeds']:
    splitter = StratifiedGroupKFold(
        n_splits=CONFIG['n_folds'],
        shuffle=True,
        random_state=seed,
    )
    for fold, (idx_train, idx_test) in enumerate(splitter.split(dummy_X, y, groups_cv)):
        assert not (set(groups_cv[idx_train]) & set(groups_cv[idx_test]))
        split_indices.append((seed, fold, idx_train, idx_test))

        for role, indices in [('train', idx_train), ('test', idx_test)]:
            for idx in indices:
                splits.append({
                    'seed': seed,
                    'fold': fold,
                    'rol': role,
                    'subject_id': subject_ids[idx],
                    'label': int(y[idx]),
                    'grupo': 'PD_OFF' if y[idx] == 1 else 'HC',
                })

splits_df = pd.DataFrame(splits)
splits_path = OUT_DIR / 'particiones.csv'
splits_df.to_csv(splits_path, index=False)

print('Particiones externas:', len(split_indices))
print('Archivo:', splits_path)
display(
    splits_df.groupby(['seed', 'fold', 'rol', 'grupo'])
    .size().rename('n').reset_index().head(20)
)


## 5. Evaluación con regresión logística

La regresión logística utiliza los mismos hiperparámetros del modelado principal (`C=1`, `class_weight='balanced'`, `max_iter=2000`).

- Para PSD discreta, escalares, A1 B-spline y A1 Cholesky, `StandardScaler` se ajusta solo con entrenamiento.
- Para **PSD discreta + PCA**, PCA se ajusta solo con entrenamiento sobre los 190 valores de `log10(PSD relativa)`; se retiene el mínimo número de componentes que explica al menos 95 % de la varianza. Los scores se estandarizan después de PCA.
- Para **A1 Cholesky + FPCA**, la PCA se ajusta de forma independiente por canal sobre las coordenadas Cholesky de entrenamiento, con el mismo criterio de 95 % usado en el cuaderno de modelado (03).

In [ ]:
# ============================================================================
# 5. HELPERS DE EVALUACIÓN
# ============================================================================
def compute_metrics(y_true: np.ndarray, y_proba: np.ndarray) -> dict[str, float]:
    """Métricas binarias a nivel de sujeto con umbral 0.5."""
    y_pred = (y_proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    try:
        auc = float(roc_auc_score(y_true, y_proba))
    except ValueError:
        auc = np.nan
    return {
        'balanced_acc': float(balanced_accuracy_score(y_true, y_pred)),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'auc': auc,
    }


def fit_logreg(X_train: np.ndarray, y_train: np.ndarray) -> LogisticRegression:
    model = LogisticRegression(
        C=CONFIG['logreg_C'],
        class_weight='balanced',
        max_iter=CONFIG['logreg_max_iter'],
        random_state=CONFIG['random_state'],
    )
    model.fit(X_train, y_train)
    return model


def scale_flatten(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Aplana y estandariza usando únicamente parámetros de entrenamiento."""
    tr = X_train.reshape(len(X_train), -1)
    te = X_test.reshape(len(X_test), -1)
    scaler = StandardScaler()
    return scaler.fit_transform(tr), scaler.transform(te)


def fpca_per_channel(
    X_train: np.ndarray,
    X_test: np.ndarray,
    variance_threshold: float = 0.95,
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    """Replica la FPCA por canal usada para A1 en el cuaderno de modelado (03)."""
    n_train, n_channels, n_basis = X_train.shape
    n_test = X_test.shape[0]
    pcas: list[PCA] = []
    n_components: list[int] = []
    explained: list[float] = []

    for channel in range(n_channels):
        pca_all = PCA().fit(X_train[:, channel, :])
        cumulative = np.cumsum(pca_all.explained_variance_ratio_)
        n_comp = int(np.searchsorted(cumulative, variance_threshold) + 1)
        n_comp = min(max(n_comp, 1), n_basis)
        pca = PCA(n_components=n_comp).fit(X_train[:, channel, :])
        pcas.append(pca)
        n_components.append(n_comp)
        explained.append(float(pca.explained_variance_ratio_.sum()))

    n_max = max(n_components)
    tr = np.zeros((n_train, n_channels, n_max), dtype=float)
    te = np.zeros((n_test, n_channels, n_max), dtype=float)
    for channel, pca in enumerate(pcas):
        n_comp = n_components[channel]
        tr[:, channel, :n_comp] = pca.transform(X_train[:, channel, :])
        te[:, channel, :n_comp] = pca.transform(X_test[:, channel, :])

    return tr, te, {
        'n_components_per_channel': n_components,
        'explained_var_per_channel': explained,
        'n_comp_max': n_max,
    }


def evaluate_method(
    method: str,
    idx_train: np.ndarray,
    idx_test: np.ndarray,
) -> tuple[dict[str, float], dict[str, Any]]:
    """Evalúa una representación sobre un pliegue externo concreto."""
    y_train, y_test = y[idx_train], y[idx_test]
    info: dict[str, Any] = {}

    if method == 'Escalar theta+alfa+PAF':
        X_train, X_test = scale_flatten(X_SCALAR[idx_train], X_SCALAR[idx_test])

    elif method == 'PSD discreta 4-13 Hz':
        X_train, X_test = scale_flatten(X_GRID[idx_train], X_GRID[idx_test])

    elif method == 'PSD discreta 4-13 Hz + PCA':
        raw_train = X_GRID[idx_train].reshape(len(idx_train), -1)
        raw_test = X_GRID[idx_test].reshape(len(idx_test), -1)
        pca = PCA(
            n_components=CONFIG['pca_variance_threshold'],
            svd_solver='full',
        )
        train_scores = pca.fit_transform(raw_train)
        test_scores = pca.transform(raw_test)
        scaler = StandardScaler()
        X_train = scaler.fit_transform(train_scores)
        X_test = scaler.transform(test_scores)
        info['n_components_pca'] = int(pca.n_components_)
        info['variance_pca'] = float(pca.explained_variance_ratio_.sum())

    elif method == 'A1 B-spline':
        X_train, X_test = scale_flatten(X_A1_RAW[idx_train], X_A1_RAW[idx_test])

    elif method == 'A1 Cholesky':
        X_train, X_test = scale_flatten(X_A1_CHOL[idx_train], X_A1_CHOL[idx_test])

    elif method == 'A1 Cholesky+FPCA':
        train_fpca, test_fpca, fpca_info = fpca_per_channel(
            X_A1_CHOL[idx_train],
            X_A1_CHOL[idx_test],
            CONFIG['pca_variance_threshold'],
        )
        X_train, X_test = scale_flatten(train_fpca, test_fpca)
        info['n_comp_max_fpca'] = int(fpca_info['n_comp_max'])
        info['n_components_fpca_total'] = int(sum(fpca_info['n_components_per_channel']))
        info['n_components_fpca_por_canal'] = json.dumps(
            fpca_info['n_components_per_channel']
        )

    else:
        raise ValueError(f'Método desconocido: {method}')

    model = fit_logreg(X_train, y_train)
    probability = model.predict_proba(X_test)[:, 1]
    return compute_metrics(y_test, probability), info


In [ ]:
# ============================================================================
# 6. EJECUCIÓN DEL ANÁLISIS DE ABLACIÓN
# ============================================================================
METHODS = [
    'Escalar theta+alfa+PAF',
    'PSD discreta 4-13 Hz',
    'PSD discreta 4-13 Hz + PCA',
    'A1 B-spline',
    'A1 Cholesky',
    'A1 Cholesky+FPCA',
]

records: list[dict[str, Any]] = []
transform_records: list[dict[str, Any]] = []

for seed, fold, idx_train, idx_test in split_indices:
    LOGGER.info('Semilla=%s | fold=%s', seed, fold)
    for method in METHODS:
        metrics, info = evaluate_method(method, idx_train, idx_test)
        records.append({
            'metodo': method,
            'seed': seed,
            'fold': fold,
            'n_train': len(idx_train),
            'n_test': len(idx_test),
            **metrics,
        })
        if info:
            transform_records.append({
                'metodo': method,
                'seed': seed,
                'fold': fold,
                **info,
            })

fold_metrics = pd.DataFrame(records)
transform_info = pd.DataFrame(transform_records)

fold_path = OUT_DIR / 'metricas_por_fold.csv'
transform_path = OUT_DIR / 'transformaciones_por_fold.csv'
fold_metrics.to_csv(fold_path, index=False)
transform_info.to_csv(transform_path, index=False)

print('Métricas por fold:', fold_path)
print('Transformaciones por fold:', transform_path)
display(fold_metrics.head(12))


## 7. Resumen y diferencias pareadas

Los 15 resultados de validación cruzada repetida **no se tratan como observaciones independientes para una prueba t convencional**. Se reportan media ± desviación estándar y diferencias descriptivas pareadas dentro de las mismas combinaciones semilla/pliegue.

La comparación central es:

\[
\text{PSD discreta 4–13 Hz}\quad \text{vs.}\quad \text{A1 B-spline}
\]

porque ambos parten del mismo `log10(PSD relativa)` y difieren principalmente en la proyección/suavizado funcional. Las comparaciones con Cholesky y FPCA permiten separar adicionalmente el efecto de la geometría funcional y de la reducción de dimensión.

In [ ]:
# ============================================================================
# 7. RESUMEN GLOBAL Y DELTAS PAREADOS
# ============================================================================
METRICS = ['balanced_acc', 'sensitivity', 'specificity', 'f1', 'auc']

summary_rows: list[dict[str, Any]] = []
for method in METHODS:
    subset = fold_metrics[fold_metrics['metodo'] == method]
    row: dict[str, Any] = {'metodo': method, 'n_evaluaciones': len(subset)}
    for metric in METRICS:
        row[f'{metric}_mean'] = float(subset[metric].mean())
        row[f'{metric}_std'] = float(subset[metric].std(ddof=1))
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / 'resumen_LogReg.csv'
summary.to_csv(summary_path, index=False)

def mean_sd(mean: float, std: float) -> str:
    return f'{mean:.3f} ± {std:.3f}'

summary_display = pd.DataFrame({
    'Método': summary['metodo'],
    'Balanced accuracy': [
        mean_sd(m, s) for m, s in zip(summary['balanced_acc_mean'], summary['balanced_acc_std'])
    ],
    'Sensibilidad': [
        mean_sd(m, s) for m, s in zip(summary['sensitivity_mean'], summary['sensitivity_std'])
    ],
    'Especificidad': [
        mean_sd(m, s) for m, s in zip(summary['specificity_mean'], summary['specificity_std'])
    ],
    'F1': [mean_sd(m, s) for m, s in zip(summary['f1_mean'], summary['f1_std'])],
    'AUC': [mean_sd(m, s) for m, s in zip(summary['auc_mean'], summary['auc_std'])],
})

print('=' * 90)
print('APORTE DEL ADF — REGRESIÓN LOGÍSTICA, 5 FOLDS × 3 SEMILLAS')
print('=' * 90)
display(summary_display)
print('Resumen:', summary_path)

# Diferencias pareadas por semilla/fold.
comparisons = [
    ('PSD discreta 4-13 Hz', 'A1 B-spline', 'A1 B-spline - PSD discreta'),
    ('A1 B-spline', 'A1 Cholesky', 'A1 Cholesky - A1 B-spline'),
    ('A1 Cholesky', 'A1 Cholesky+FPCA', 'A1 Cholesky+FPCA - A1 Cholesky'),
    ('PSD discreta 4-13 Hz', 'PSD discreta 4-13 Hz + PCA', 'PSD+PCA - PSD discreta'),
    ('Escalar theta+alfa+PAF', 'A1 B-spline', 'A1 B-spline - Escalar theta+alfa+PAF'),
]

delta_rows: list[dict[str, Any]] = []
for base_method, comparison_method, label in comparisons:
    base = fold_metrics[fold_metrics['metodo'] == base_method].set_index(['seed', 'fold'])
    comp = fold_metrics[fold_metrics['metodo'] == comparison_method].set_index(['seed', 'fold'])
    common = base.index.intersection(comp.index)
    for seed, fold in common:
        row = {
            'comparacion': label,
            'seed': seed,
            'fold': fold,
        }
        for metric in METRICS:
            row[f'delta_{metric}'] = float(comp.loc[(seed, fold), metric] - base.loc[(seed, fold), metric])
        delta_rows.append(row)

paired_deltas = pd.DataFrame(delta_rows)
delta_path = OUT_DIR / 'deltas_pareados.csv'
paired_deltas.to_csv(delta_path, index=False)

delta_summary_rows = []
for comparison, subset in paired_deltas.groupby('comparacion', sort=False):
    row = {'comparacion': comparison, 'n': len(subset)}
    for metric in METRICS:
        values = subset[f'delta_{metric}']
        row[f'delta_{metric}_mean'] = float(values.mean())
        row[f'delta_{metric}_std'] = float(values.std(ddof=1))
        row[f'delta_{metric}_min'] = float(values.min())
        row[f'delta_{metric}_max'] = float(values.max())
    delta_summary_rows.append(row)

delta_summary = pd.DataFrame(delta_summary_rows)
delta_summary_path = OUT_DIR / 'resumen_deltas_pareados.csv'
delta_summary.to_csv(delta_summary_path, index=False)

print('\nDiferencias pareadas descriptivas:')
display(delta_summary[[
    'comparacion',
    'delta_balanced_acc_mean', 'delta_balanced_acc_std',
    'delta_auc_mean', 'delta_auc_std',
]].round(4))
print('Deltas por fold:', delta_path)
print('Resumen de deltas:', delta_summary_path)


## 8. Verificación de consistencia con el cuaderno de modelado

Se comprueba que este cuaderno reproduce la exactitud balanceada de la regresión logística de A1
obtenida en el cuaderno de modelado (03), antes de interpretar los controles. Diferencias pequeñas
pueden aparecer por versiones de bibliotecas; desviaciones mayores a 0,01 indicarían una
discrepancia en el procedimiento que debe revisarse.

In [ ]:
# ============================================================================
# 8. CONSISTENCIA CON EL CUADERNO DE MODELADO (A1, LogReg)
# ============================================================================
# Exactitud balanceada de LogReg en A1 reportada por el cuaderno 03.
A1_BALACC_MODELADO = {
    'A1 B-spline': 0.792,
    'A1 Cholesky': 0.818,
    'A1 Cholesky+FPCA': 0.803,
}

trace_rows = []
for method, reference_value in A1_BALACC_MODELADO.items():
    current = float(summary.loc[summary['metodo'] == method, 'balanced_acc_mean'].iloc[0])
    trace_rows.append({
        'metodo': method,
        'balanced_acc_modelado': reference_value,
        'balanced_acc_actual': current,
        'diferencia': current - reference_value,
        'coincide_tolerancia_0_01': abs(current - reference_value) <= 0.01,
    })

trace = pd.DataFrame(trace_rows)
trace_path = OUT_DIR / 'consistencia_A1_modelado.csv'
trace.to_csv(trace_path, index=False)
display(trace.round(4))

if not trace['coincide_tolerancia_0_01'].all():
    print('ADVERTENCIA: alguna configuración A1 difiere más de 0.01 del cuaderno de modelado. Revisar el procedimiento.')
else:
    print('Consistencia con el cuaderno de modelado dentro de la tolerancia ±0.01.')


## 9. Figura resumen

La figura muestra media ± desviación estándar sobre las 15 evaluaciones externas. Debe interpretarse
de forma descriptiva; las barras de variabilidad no constituyen intervalos de confianza
independientes porque los pliegues de validación cruzada repetida están correlacionados.

In [ ]:
# ============================================================================
# 9. FIGURA RESUMEN
# ============================================================================
plot_labels = {
    'Escalar theta+alfa+PAF': 'Escalar theta+alfa+PAF',
    'PSD discreta 4-13 Hz': 'PSD discreta 4–13 Hz',
    'PSD discreta 4-13 Hz + PCA': 'PSD discreta + PCA',
    'A1 B-spline': 'A1 B-spline',
    'A1 Cholesky': 'A1 Cholesky',
    'A1 Cholesky+FPCA': 'A1 Cholesky+FPCA',
}

plot_df = summary.set_index('metodo').loc[METHODS].reset_index()
y_pos = np.arange(len(plot_df))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), sharey=True)
for ax, metric, xlabel in [
    (axes[0], 'balanced_acc', 'Exactitud balanceada'),
    (axes[1], 'auc', 'AUC'),
]:
    means = plot_df[f'{metric}_mean'].to_numpy()
    stds = plot_df[f'{metric}_std'].to_numpy()
    ax.errorbar(means, y_pos, xerr=stds, fmt='o', capsize=4, lw=1.3)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.20)
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_yticks(y_pos)
axes[0].set_yticklabels([plot_labels[m] for m in plot_df['metodo']])
axes[0].invert_yaxis()
fig.tight_layout()

figure_path = FIG_DIR / 'comparacion_representaciones_LogReg.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura guardada:', figure_path)


## 10. Interpretación de los resultados

- **PSD discreta 4–13 Hz vs. A1 B-spline:** ambas parten del mismo `log10(PSD relativa)`. La PSD
  discreta obtuvo una exactitud balanceada de 0,818 ± 0,117 y A1 B-spline de 0,792 ± 0,114
  (diferencia pareada media de −0,026). En esta comparación controlada no se observa una ganancia
  predictiva atribuible exclusivamente a la proyección B-spline.
- **A1 B-spline → A1 Cholesky:** el cambio de coordenadas que preserva la geometría L2 llevó a
  0,818 ± 0,109 (AUC 0,879 ± 0,097), prácticamente equivalente a la PSD discreta.
- **Reducción de dimensión:** ni FPCA sobre A1 Cholesky (0,803 ± 0,138) ni PCA sobre la PSD
  discreta (0,798 ± 0,126) produjeron mejora adicional.
- **Theta + alfa + PAF** (0,758 ± 0,143) indica que parte de la información discriminante está en
  la forma detallada del espectro 4–13 Hz y no queda completamente representada por un conjunto
  reducido de características escalares.

Con regresión logística en A1 no se observa una ventaja predictiva adicional
atribuible exclusivamente al ADF. La representación funcional aporta una base de menor dimensión,
la incorporación explícita de la geometría funcional y la posibilidad de interpretar el clasificador
sobre el dominio continuo de la frecuencia (cuaderno 05).

Las diferencias se interpretan de forma descriptiva: no se aplican pruebas de hipótesis tratando
las 15 evaluaciones como observaciones independientes.

In [ ]:
# ============================================================================
# 10. VERIFICACIÓN DE LOS PRODUCTOS GENERADOS
# ============================================================================
expected_files = [
    CONFIG_PATH,
    FEATURE_PATH,
    splits_path,
    fold_path,
    transform_path,
    summary_path,
    delta_path,
    delta_summary_path,
    trace_path,
    figure_path,
]

for path in expected_files:
    if not path.exists():
        raise FileNotFoundError(f'No se generó el producto esperado: {path}')

assert len(fold_metrics) == len(METHODS) * 15
assert fold_metrics.groupby('metodo').size().eq(15).all()
assert summary['n_evaluaciones'].eq(15).all()
assert np.isfinite(fold_metrics[METRICS].to_numpy()).all()

run_metadata = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'n_subjects': int(len(subject_ids)),
    'class_counts': {'HC': int((y == 0).sum()), 'PD_OFF': int((y == 1).sum())},
    'n_external_evaluations': 15,
    'methods': METHODS,
    'feature_shapes': {
        'scalar_theta_alpha_paf': list(X_SCALAR.shape),
        'psd_grid_4_13': list(X_GRID.shape),
        'A1_raw': list(X_A1_RAW.shape),
        'A1_cholesky': list(X_A1_CHOL.shape),
    },
    'environment': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'scipy': scipy.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
    },
}
metadata_path = OUT_DIR / 'metadatos_ejecucion.json'
metadata_path.write_text(json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding='utf-8')

print('=' * 90)
print('CUADERNO 04 COMPLETADO: PRODUCTOS VERIFICADOS')
print('=' * 90)
for path in expected_files + [metadata_path]:
    print(' -', path)

